# Team pressing — Step 2: build-up episodes
Barca vs Atletico — first half. Self-contained: reads only the two frame tables and recomputes the pressure signal with the step-1 config (unchanged thresholds), so it does not depend on any step-1 output.

**Episode** = one possession of the build-up team (A) inside A's build-up zone (own goal line → 70% of the pitch).

| | Rule |
|---|---|
| Zone | `own_goal_distance(ball) <= 0.70 * PITCH_LENGTH` (73.5 m), measured from A's own goal, penalty box included |
| Start | first frame where A has possession and the ball is inside the zone |
| End: `escaped` | ball stays beyond the 70% line for >= 0.5 s while A still has possession |
| End: `lost` | opponent holds possession for >= 0.5 s |
| End: `stopped` / `data_end` | stoppage >= 10 s (v30 detection) / end of data |
| Cleaning | episodes < 1.0 s dropped; the 0.5 s holds absorb deflections and boundary jitter |

**Pressure inside an episode** is *measured*, not part of the definition: an episode is `pressed` if it overlaps at least one v30 pressure event (computed below, unchanged thresholds) of the defending team. The defending team's own box can never fall inside the zone, so v30's own-box exclusion has no effect here.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path

FPS = 25
PITCH_LENGTH = 105
PITCH_WIDTH = 68

CACHE = Path(r"C:\Users\user\Desktop\Football_cv_project\football-pipeline (1)"
             r"\project\barca_atletico_first_half\new_cache")
PLAYER_PATH = CACHE / "player_frame_table_named.parquet"
BALL_PATH = CACHE / "ball_frame_table_named.parquet"

player_frame_table = pd.read_parquet(PLAYER_PATH)
ball_frame_table = pd.read_parquet(BALL_PATH)

# --- adapt to the *_named table structure -------------------------------------
# 1) nullable Int64 -> float64, so <NA> behaves as NaN in comparisons / np.where / groupby
player_frame_table["team"] = player_frame_table["team"].astype("float64")
for c in ["carrier_track_id", "carrier_team", "possession_team"]:
    ball_frame_table[c] = ball_frame_table[c].astype("float64")

# 2) the player table now ALSO carries carrier_track_id / possession_team. The ball table stays the
#    single source of truth (as in v30), so drop the player-side copies -> no _x/_y merge collisions.
player_frame_table = player_frame_table.drop(
    columns=[c for c in ["carrier_track_id", "possession_team"] if c in player_frame_table.columns]
)

# 3) team ids are NOT hard-coded any more. In this cache BAR is team 1 (v30 assumed Barca = team 0),
#    so which team defends x=0 is inferred from the table's own dist_from_own_goal column.
_chk = player_frame_table[
    (player_frame_table["role"] == "player")
    & player_frame_table[["pitch_x", "dist_from_own_goal", "team"]].notna().all(axis=1)
]
_own_goal_at_zero = {}
for _t, _g in _chk.groupby("team"):
    _d_zero = (_g["pitch_x"] - _g["dist_from_own_goal"]).abs().median()
    _d_far = (_g["pitch_x"] + _g["dist_from_own_goal"] - PITCH_LENGTH).abs().median()
    _own_goal_at_zero[int(_t)] = bool(_d_zero < _d_far)
_zero_teams = [t for t, v in _own_goal_at_zero.items() if v]
assert len(_zero_teams) == 1, f"expected exactly one team defending x=0, got {_own_goal_at_zero}"
ZERO_TEAM = _zero_teams[0]   # team whose own goal is at x = 0; the other team defends x = PITCH_LENGTH

def own_goal_distance(team, x):
    # metres from the pressing/defending team's own goal line (v30: np.where(team == 0, x, L - x))
    return np.where(np.asarray(team) == ZERO_TEAM, np.asarray(x, dtype=float), PITCH_LENGTH - np.asarray(x, dtype=float))

TEAM_NAMES = (
    player_frame_table.dropna(subset=["team", "team_name"])
    .groupby("team")["team_name"].agg(lambda s: s.mode().iloc[0])
    .rename(index=int).to_dict()
)

# --- sanity checks -------------------------------------------------------------
print("team ids -> names:", TEAM_NAMES, "| team defending x=0:", ZERO_TEAM, f"({TEAM_NAMES[ZERO_TEAM]})")
print("attack_dir by team:", player_frame_table.dropna(subset=["team", "attack_dir"]).groupby("team")["attack_dir"].unique().to_dict())
print("roles:", player_frame_table["role"].value_counts().to_dict())
print("duplicate (frame_idx, track_id) rows:", int(player_frame_table.duplicated(["frame_idx", "track_id"]).sum()))
print(f"player rows with valid pitch coords: {player_frame_table['pitch_x'].notna().mean():.1%} | "
      f"ball frames with carrier: {ball_frame_table['carrier_track_id'].notna().mean():.1%}")
print("frames:", ball_frame_table["frame_idx"].nunique(), "| player frames:", player_frame_table["frame_idx"].nunique())


## 1. Team defensive baseline
Average depth of each team's back line (last-3 defenders), computed once up front, restricted to frames where that team is out of possession (their actual defending shape, not inflated by attacking-phase positioning). Used in the proximity step to tell "stepped out to press" apart from "block was already sitting there".

In [ ]:
LINE_HEIGHT_DEFENDERS_N = 3

_poss_for_baseline = ball_frame_table[["frame_idx", "possession_team"]]

_outfield_baseline = player_frame_table[player_frame_table["role"] == "player"].dropna(subset=["pitch_x", "pitch_y"]).copy()
_outfield_baseline = _outfield_baseline.merge(_poss_for_baseline, on="frame_idx", how="left")
_outfield_baseline = _outfield_baseline[
    _outfield_baseline["possession_team"].notna() &
    (_outfield_baseline["team"] != _outfield_baseline["possession_team"])
]  # keep only frames where this team is out of possession -- their real defensive shape

_outfield_baseline["dist_from_own_goal"] = own_goal_distance(_outfield_baseline["team"], _outfield_baseline["pitch_x"])
_outfield_baseline["dist_from_own_goal"] = _outfield_baseline["dist_from_own_goal"].clip(0, PITCH_LENGTH)

team_defensive_line_avg = (
    _outfield_baseline.sort_values("dist_from_own_goal")
    .groupby(["frame_idx", "team"])
    .head(LINE_HEIGHT_DEFENDERS_N)
    .groupby("team")["dist_from_own_goal"]
    .mean()
    .to_dict()
)
print("team average defensive line depth, out-of-possession only (last-3 defenders, m from own goal):", team_defensive_line_avg)

## 2. Proximity pressure — distance from each off-ball defender to the ball carrier
A defender only counts as "pressuring" if they're close AND (a) they're closing the distance fast (actively running at the carrier, >=2.5 m/s), or (b) they're within touch-tight range (<1.5m) regardless of speed, or (c) they're clearly ahead of their team's own out-of-possession baseline line (stepped out of the block). This is meant to separate real pressing from a deep, compact block that's simply already sitting near the ball because the attacker walked into it. Thresholds tightened from the first pass -- 1.0 m/s and 2.0m were too permissive and let most of a passive block still count as "pressuring".

In [ ]:
PRESSURE_DIST_THRESH_M = 5.0
CLOSING_WINDOW_FRAMES = 12       # ~0.5s @ 25fps, window used to measure closing speed
CLOSING_SPEED_THRESH_MPS = 2.5   # must be closing at least this fast to count as an active press from open play (walking pace was too permissive)
TIGHT_MARK_DIST_M = 1.5          # inside this range, counts as pressure even without active closing (touch-tight)
STEP_OUT_MARGIN_M = 4.0          # must be at least this far ahead of the team's own average line depth

carrier_pos = ball_frame_table[
    ["frame_idx", "carrier_track_id", "carrier_team", "possession_team", "ball_pitch_x", "ball_pitch_y"]
]

pft = player_frame_table.merge(carrier_pos, on="frame_idx", how="left")

off_ball = pft[
    (pft["role"] == "player") &
    pft["carrier_track_id"].notna() &
    (pft["track_id"] != pft["carrier_track_id"]) &
    (pft["team"] != pft["carrier_team"])
].copy()

off_ball["dist_to_carrier"] = (
    (off_ball["pitch_x"] - off_ball["ball_pitch_x"]) ** 2 +
    (off_ball["pitch_y"] - off_ball["ball_pitch_y"]) ** 2
) ** 0.5

off_ball_pressure = off_ball.dropna(subset=["dist_to_carrier"]).copy()
off_ball_pressure = off_ball_pressure.sort_values(["track_id", "frame_idx"]).reset_index(drop=True)

# closing speed: how fast dist_to_carrier has been shrinking over the last CLOSING_WINDOW_FRAMES for this player
off_ball_pressure["dist_to_carrier_prev"] = off_ball_pressure.groupby("track_id")["dist_to_carrier"].shift(CLOSING_WINDOW_FRAMES)
off_ball_pressure["frame_gap"] = off_ball_pressure["frame_idx"] - off_ball_pressure.groupby("track_id")["frame_idx"].shift(CLOSING_WINDOW_FRAMES)
off_ball_pressure["closing_speed_mps"] = np.where(
    (off_ball_pressure["frame_gap"] > 0) & (off_ball_pressure["frame_gap"] <= CLOSING_WINDOW_FRAMES * 2),
    (off_ball_pressure["dist_to_carrier_prev"] - off_ball_pressure["dist_to_carrier"]) / (off_ball_pressure["frame_gap"] / FPS),
    np.nan
)

# stepped ahead of the team's own habitual defensive line -> not just standing in the block
off_ball_pressure["dist_from_own_goal"] = own_goal_distance(off_ball_pressure["team"], off_ball_pressure["pitch_x"])
off_ball_pressure["dist_from_own_goal"] = off_ball_pressure["dist_from_own_goal"].clip(0, PITCH_LENGTH)
off_ball_pressure["team_line_avg"] = off_ball_pressure["team"].map(team_defensive_line_avg)
off_ball_pressure["stepped_out_of_line"] = (
    off_ball_pressure["dist_from_own_goal"] >= off_ball_pressure["team_line_avg"] + STEP_OUT_MARGIN_M
)

off_ball_pressure["is_pressuring"] = (
    (off_ball_pressure["dist_to_carrier"] < PRESSURE_DIST_THRESH_M) &
    (
        (off_ball_pressure["dist_to_carrier"] < TIGHT_MARK_DIST_M) |
        (off_ball_pressure["closing_speed_mps"].fillna(0) >= CLOSING_SPEED_THRESH_MPS) |
        off_ball_pressure["stepped_out_of_line"]
    )
)

team_pressure_per_frame = (
    off_ball_pressure.groupby(["frame_idx", "team", "carrier_team", "possession_team"], as_index=False)
    .agg(
        n_pressing=("is_pressuring", "sum"),
        n_defenders_on_pitch=("track_id", "nunique"),
        min_dist_to_carrier=("dist_to_carrier", "min"),
    )
)
team_pressure_per_frame["pressing_intensity"] = (
    team_pressure_per_frame["n_pressing"] / team_pressure_per_frame["n_defenders_on_pitch"]
)

## 3. Passing-lane pressure — how covered are the carrier's passing options

In [ ]:
MIN_LANE_LENGTH_M = 3.0
TIGHT_LANE_THRESH_M = 1.5
CONTESTED_LANE_THRESH_M = 3.0

carrier_frames = pft[(pft["role"] == "player") & pft["carrier_track_id"].notna()].copy()

teammates = carrier_frames[
    (carrier_frames["team"] == carrier_frames["carrier_team"]) &
    (carrier_frames["track_id"] != carrier_frames["carrier_track_id"])
]
opponents = carrier_frames[carrier_frames["team"] != carrier_frames["carrier_team"]]

carrier_player_pos = (
    player_frame_table[player_frame_table["is_carrier"] == True]
    [["frame_idx", "track_id", "pitch_x", "pitch_y"]]
    .rename(columns={"track_id": "carrier_track_id", "pitch_x": "carrier_x", "pitch_y": "carrier_y"})
    .drop_duplicates(subset="frame_idx", keep="first")
)

teammate_pos = teammates[["frame_idx", "track_id", "pitch_x", "pitch_y"]].rename(
    columns={"track_id": "teammate_id", "pitch_x": "tm_x", "pitch_y": "tm_y"}
)
opponent_pos = opponents[["frame_idx", "track_id", "pitch_x", "pitch_y"]].rename(
    columns={"track_id": "opponent_id", "pitch_x": "opp_x", "pitch_y": "opp_y"}
)

lanes = teammate_pos.merge(opponent_pos, on="frame_idx", how="inner")
lanes = lanes.merge(carrier_player_pos, on="frame_idx", how="inner")
lanes = lanes.dropna(subset=["carrier_x", "carrier_y"])

Ax, Ay = lanes["carrier_x"].values, lanes["carrier_y"].values
Bx, By = lanes["tm_x"].values, lanes["tm_y"].values
Px, Py = lanes["opp_x"].values, lanes["opp_y"].values

ABx, ABy = Bx - Ax, By - Ay
APx, APy = Px - Ax, Py - Ay

seg_len_sq = ABx ** 2 + ABy ** 2
seg_len_sq = np.where(seg_len_sq == 0, 1e-9, seg_len_sq)

t = (APx * ABx + APy * ABy) / seg_len_sq
t_clipped = np.clip(t, 0.0, 1.0)

closest_x = Ax + t_clipped * ABx
closest_y = Ay + t_clipped * ABy
perp_dist = np.sqrt((Px - closest_x) ** 2 + (Py - closest_y) ** 2)

lanes["t"] = t
lanes["lane_perp_dist"] = perp_dist
lanes["lane_length_m"] = np.sqrt(seg_len_sq)

valid_lanes = lanes[
    (lanes["lane_length_m"] >= MIN_LANE_LENGTH_M) &
    (lanes["t"] >= 0.15) &
    (lanes["t"] <= 0.85)
].copy()

valid_lanes["lane_tight_block"] = valid_lanes["lane_perp_dist"] < TIGHT_LANE_THRESH_M
valid_lanes["lane_contested"] = valid_lanes["lane_perp_dist"] < CONTESTED_LANE_THRESH_M

lane_status = (
    valid_lanes.groupby(["frame_idx", "teammate_id"], as_index=False)
    .agg(lane_tight_block=("lane_tight_block", "any"), lane_contested=("lane_contested", "any"))
)

frame_lane_summary = (
    lane_status.groupby("frame_idx", as_index=False)
    .agg(
        n_lanes_total=("teammate_id", "nunique"),
        n_lanes_tight_blocked=("lane_tight_block", "sum"),
        n_lanes_contested=("lane_contested", "sum"),
    )
)
frame_lane_summary["n_lanes_open"] = frame_lane_summary["n_lanes_total"] - frame_lane_summary["n_lanes_contested"]
frame_lane_summary["frac_tight_blocked"] = frame_lane_summary["n_lanes_tight_blocked"] / frame_lane_summary["n_lanes_total"]
frame_lane_summary["frac_contested"] = frame_lane_summary["n_lanes_contested"] / frame_lane_summary["n_lanes_total"]

frame_lane_summary = frame_lane_summary.merge(
    carrier_pos[["frame_idx", "carrier_team", "possession_team"]], on="frame_idx", how="left"
)


## 4. Unified team pressure signal + events
Combines proximity pressure OR lane-blocking pressure, then excludes own-penalty-box frames (16.5m from own goal) since compact box defending is a different phenomenon from active pressing. This corrected `under_team_pressure` definition is used everywhere downstream (events, zone breakdown, time-to-press, pass resistance, weighted intensity).
Note: `n_pressing`/`pressing_intensity` now come from the closing-speed + step-out-of-line gated definition in section 1, not raw distance -- so a passive deep block no longer inflates this on its own. The lane-blocking signal (section 2) is left distance-based for now, since being within `TIGHT_LANE_THRESH_M` of an actual pass lane is a real obstruction regardless of how the defender got there.

In [ ]:
N_PRESSING_THRESH = 2
FRAC_TIGHT_BLOCKED_THRESH = 0.375
MIN_EVENT_DURATION_SEC = 0.5
GAP_TOLERANCE_FRAMES = 5
BOX_DEPTH_M = 16.5

team_pressure_full = team_pressure_per_frame.merge(
    frame_lane_summary[["frame_idx", "n_lanes_total", "n_lanes_tight_blocked",
                         "n_lanes_contested", "frac_tight_blocked", "frac_contested"]],
    on="frame_idx",
    how="left"
)

ball_pos_by_frame = ball_frame_table[["frame_idx", "ball_pitch_x", "ball_pitch_y"]].rename(
    columns={"ball_pitch_x": "press_x", "ball_pitch_y": "press_y"}
)
team_pressure_full = team_pressure_full.merge(ball_pos_by_frame, on="frame_idx", how="left")

tpf = team_pressure_full.copy()
tpf["frac_tight_blocked"] = tpf["frac_tight_blocked"].fillna(0)

tpf["in_own_box"] = own_goal_distance(tpf["team"], tpf["press_x"]) <= BOX_DEPTH_M

tpf["under_team_pressure"] = (
    ((tpf["n_pressing"] >= N_PRESSING_THRESH) | (tpf["frac_tight_blocked"] >= FRAC_TIGHT_BLOCKED_THRESH)) &
    (~tpf["in_own_box"])
)

# carry the corrected flag back onto team_pressure_full so every downstream section uses the same definition
team_pressure_full["under_team_pressure"] = tpf["under_team_pressure"]

tpf = tpf.sort_values(["team", "frame_idx"]).reset_index(drop=True)

events_list = []
for team_id, grp in tpf.groupby("team"):
    grp = grp.sort_values("frame_idx").reset_index(drop=True)

    is_press = grp["under_team_pressure"].values.copy()
    frames = grp["frame_idx"].values
    i, n = 0, len(is_press)
    while i < n:
        if not is_press[i]:
            j = i
            while j < n and not is_press[j]:
                j += 1
            gap_frame_span = frames[j - 1] - frames[i] + 1 if j > i else 0
            if i > 0 and j < n and gap_frame_span <= GAP_TOLERANCE_FRAMES:
                is_press[i:j] = True
            i = j
        else:
            i += 1
    grp["under_team_pressure_bridged"] = is_press

    grp["state_change"] = (grp["under_team_pressure_bridged"] != grp["under_team_pressure_bridged"].shift()).cumsum()
    runs = (
        grp.groupby("state_change")
        .agg(
            start_frame=("frame_idx", "first"),
            end_frame=("frame_idx", "last"),
            n_frames=("frame_idx", "count"),
            under_pressure=("under_team_pressure_bridged", "first"),
            possession_team=("possession_team", "first"),
            mean_n_pressing=("n_pressing", "mean"),
            mean_frac_tight_blocked=("frac_tight_blocked", "mean"),
        )
        .reset_index(drop=True)
    )
    runs["team"] = team_id
    events_list.append(runs)

team_pressure_events = pd.concat(events_list, ignore_index=True)
team_pressure_events = team_pressure_events[team_pressure_events["under_pressure"]].copy()
team_pressure_events["duration_sec"] = team_pressure_events["n_frames"] / FPS
team_pressure_events = team_pressure_events[
    team_pressure_events["duration_sec"] >= MIN_EVENT_DURATION_SEC
].reset_index(drop=True)

print("total team pressure events:", len(team_pressure_events))
print(team_pressure_events.groupby("team")["duration_sec"].describe())


### Pressure signal ready
`team_pressure_full` (per frame, per defending team) and `team_pressure_events` (v30 events) are now computed from the tables above. Team ids are cast to int for the joins below.

In [ ]:
team_pressure_events["team"] = team_pressure_events["team"].astype(int)
team_pressure_full["team"] = team_pressure_full["team"].astype(int)
print(f"{len(team_pressure_events)} pressure events | {len(team_pressure_full)} per-frame pressure rows")

## Stoppages (v30, unchanged)

In [ ]:
STOPPAGE_GAP_SEC = 10.0
STOPPAGE_GAP_FRAMES = int(STOPPAGE_GAP_SEC * FPS)

carrier_missing = ball_frame_table[["frame_idx", "carrier_track_id"]].sort_values("frame_idx").reset_index(drop=True)
carrier_missing["is_missing"] = carrier_missing["carrier_track_id"].isna()
carrier_missing["gap_id"] = (carrier_missing["is_missing"] != carrier_missing["is_missing"].shift()).cumsum()

gap_runs = (
    carrier_missing[carrier_missing["is_missing"]]
    .groupby("gap_id")
    .agg(start_frame=("frame_idx", "first"), end_frame=("frame_idx", "last"), n_frames=("frame_idx", "count"))
    .reset_index(drop=True)
)
gap_runs["duration_sec"] = gap_runs["n_frames"] / FPS

stoppages = gap_runs[gap_runs["duration_sec"] >= STOPPAGE_GAP_SEC].reset_index(drop=True)

# vectorized frame -> is_stoppage lookup
_stoppage_starts = stoppages["start_frame"].values
_stoppage_ends = stoppages["end_frame"].values

all_frames = ball_frame_table[["frame_idx"]].drop_duplicates().sort_values("frame_idx").reset_index(drop=True)
_frame_vals = all_frames["frame_idx"].values
_idx = np.searchsorted(_stoppage_starts, _frame_vals, side="right") - 1
_valid = _idx >= 0
_in_range = np.zeros(len(_frame_vals), dtype=bool)
_idx_valid = _idx[_valid]
_in_range[_valid] = _frame_vals[_valid] <= _stoppage_ends[_idx_valid]
all_frames["is_stoppage"] = _in_range

stoppage_frame_lookup = all_frames.set_index("frame_idx")["is_stoppage"]

print(f"{len(stoppages)} stoppages >= {STOPPAGE_GAP_SEC}s flagged, covering {stoppages['n_frames'].sum()} frames "
      f"({stoppages['duration_sec'].sum():.1f}s total dead time)")

## Episode config + per-frame state
`possession_team` is forward-filled over no-carrier gaps (same keep-last rule as v30 section 12), so a ball in flight stays with the passer until someone else actually gets it.

In [ ]:
ZONE_FRAC = 0.70
ZONE_M = ZONE_FRAC * PITCH_LENGTH          # 73.5 m from the build-up team's own goal
HOLD_SEC = 0.5                             # escape / loss must persist this long
HOLD_FRAMES = int(HOLD_SEC * FPS)
MIN_EPISODE_SEC = 1.0
BALL_FFILL_FRAMES = FPS                    # bridge <= 1 s of missing ball position

ball = (
    ball_frame_table[["frame_idx", "possession_team", "ball_pitch_x"]]
    .sort_values("frame_idx").reset_index(drop=True)
    .merge(stoppage_frame_lookup.rename("is_stoppage"), left_on="frame_idx", right_index=True, how="left")
)
ball["is_stoppage"] = ball["is_stoppage"].fillna(False).astype(bool)
ball["poss"] = ball["possession_team"].ffill()
ball["ball_x"] = ball["ball_pitch_x"].ffill(limit=BALL_FFILL_FRAMES)

print(f"zone line: {ZONE_M:.1f} m from each team's own goal")
print("non-contiguous frame steps:", int((ball["frame_idx"].diff().dropna() != 1).sum()))
print(f"ball position known: {ball['ball_x'].notna().mean():.1%} | possession known: {ball['poss'].notna().mean():.1%}")

## Detect episodes

In [ ]:
def detect_episodes(team):
    opp = [t for t in TEAM_NAMES if t != team][0]
    frames = ball["frame_idx"].values
    poss = ball["poss"].values
    stop = ball["is_stoppage"].values
    dist = own_goal_distance(np.full(len(ball), team), ball["ball_x"].values)

    out, in_ep = [], False
    s = esc_run = loss_run = esc_start = loss_start = 0

    def close(e, outcome):
        out.append(dict(
            team=team, defending_team=opp,
            start_frame=int(frames[s]), end_frame=int(frames[e]),
            n_frames=int(frames[e] - frames[s] + 1), outcome=outcome,
            start_dist_m=float(dist[s]), max_dist_m=float(np.nanmax(dist[s:e + 1])),
        ))

    for i in range(len(ball)):
        if stop[i]:
            if in_ep:
                close(i - 1, "stopped"); in_ep = False
            continue
        a_has, o_has = poss[i] == team, poss[i] == opp
        if not in_ep:
            if a_has and dist[i] <= ZONE_M:
                in_ep, s, esc_run, loss_run = True, i, 0, 0
            continue
        beyond = a_has and dist[i] > ZONE_M
        esc_run = esc_run + 1 if beyond else 0
        loss_run = loss_run + 1 if o_has else 0
        if esc_run == 1: esc_start = i
        if loss_run == 1: loss_start = i
        if esc_run >= HOLD_FRAMES:
            close(esc_start - 1, "escaped"); in_ep = False
        elif loss_run >= HOLD_FRAMES:
            close(loss_start - 1, "lost"); in_ep = False
    if in_ep:
        close(len(ball) - 1, "data_end")
    return out

episodes = pd.DataFrame([e for t in TEAM_NAMES for e in detect_episodes(t)])
n_raw = len(episodes)
episodes["duration_sec"] = episodes["n_frames"] / FPS
episodes = episodes[episodes["duration_sec"] >= MIN_EPISODE_SEC].sort_values("start_frame").reset_index(drop=True)
episodes.insert(0, "episode_id", episodes.index)
episodes["team_name"] = episodes["team"].map(TEAM_NAMES)
episodes["defending_team_name"] = episodes["defending_team"].map(TEAM_NAMES)
_mmss = lambda sec: f"{int(sec // 60):02d}:{int(sec % 60):02d}"
episodes["start_mmss"] = (episodes["start_frame"] / FPS).map(_mmss)
episodes["end_mmss"] = ((episodes["end_frame"] + 1) / FPS).map(_mmss)

print(f"{n_raw} raw episodes -> {len(episodes)} after dropping < {MIN_EPISODE_SEC}s")
display(episodes.groupby("team_name").agg(
    n_episodes=("episode_id", "count"),
    mean_sec=("duration_sec", "mean"),
    median_sec=("duration_sec", "median"),
    max_sec=("duration_sec", "max"),
).round(2))
display(episodes.groupby(["team_name", "outcome"]).size().unstack(fill_value=0))

## Pressure inside each episode
- `pressed` / `n_press_events` / `pressed_sec` / `time_to_first_press_sec`: overlap with the defending team's v30 pressure events.
- `raw_frames_*`: per-frame breakdown of *why* a frame counted as pressure (proximity = `n_pressing >= 2`, lane = `frac_tight_blocked >= 0.375`). Frames with no ball carrier have no pressure row and count as not pressed.

In [ ]:
# --- v30 event overlap ---
ev_by_team = {t: g[["start_frame", "end_frame"]].values for t, g in team_pressure_events.groupby("team")}

def event_overlap(row):
    evs = ev_by_team.get(row.defending_team, np.empty((0, 2), dtype=int))
    hit = evs[(evs[:, 1] >= row.start_frame) & (evs[:, 0] <= row.end_frame)]
    if len(hit) == 0:
        return pd.Series(dict(n_press_events=0, pressed_frames=0, first_press_frame=np.nan))
    lo = np.maximum(hit[:, 0], row.start_frame)
    hi = np.minimum(hit[:, 1], row.end_frame)
    return pd.Series(dict(n_press_events=len(hit), pressed_frames=int((hi - lo + 1).sum()), first_press_frame=float(lo.min())))

episodes = episodes.drop(columns=["n_press_events", "pressed_frames", "first_press_frame"], errors="ignore")
episodes = episodes.join(episodes.apply(event_overlap, axis=1))
episodes[["n_press_events", "pressed_frames"]] = episodes[["n_press_events", "pressed_frames"]].astype(int)
episodes["pressed"] = episodes["n_press_events"] > 0
episodes["pressed_sec"] = episodes["pressed_frames"] / FPS
episodes["pct_time_pressed"] = episodes["pressed_frames"] / episodes["n_frames"]
episodes["time_to_first_press_sec"] = (episodes["first_press_frame"] - episodes["start_frame"]) / FPS

# --- per-frame breakdown ---
ep_frames = pd.concat(
    [pd.DataFrame({"episode_id": r.episode_id, "defending_team": r.defending_team,
                   "frame_idx": np.arange(r.start_frame, r.end_frame + 1)}) for r in episodes.itertuples()],
    ignore_index=True,
)
tp = team_pressure_full[["frame_idx", "team", "n_pressing", "frac_tight_blocked", "under_team_pressure"]].copy()
tp["frac_tight_blocked"] = tp["frac_tight_blocked"].fillna(0)
ep_frames = ep_frames.merge(tp, left_on=["frame_idx", "defending_team"], right_on=["frame_idx", "team"], how="left")

under = ep_frames["under_team_pressure"].fillna(False).astype(bool)
prox = ep_frames["n_pressing"].fillna(0) >= N_PRESSING_THRESH
lane = ep_frames["frac_tight_blocked"] >= FRAC_TIGHT_BLOCKED_THRESH
ep_frames["f_under"] = under
ep_frames["f_prox_only"] = under & prox & ~lane
ep_frames["f_lane_only"] = under & ~prox & lane
ep_frames["f_both"] = under & prox & lane

frame_stats = ep_frames.groupby("episode_id").agg(
    raw_frames_under_pressure=("f_under", "sum"),
    raw_frames_prox_only=("f_prox_only", "sum"),
    raw_frames_lane_only=("f_lane_only", "sum"),
    raw_frames_both=("f_both", "sum"),
    peak_n_pressing=("n_pressing", "max"),
    frames_with_carrier=("n_pressing", "count"),
).reset_index()
episodes = episodes.drop(columns=[c for c in frame_stats.columns if c != "episode_id"], errors="ignore").merge(frame_stats, on="episode_id", how="left")

print(f"{int(episodes['pressed'].sum())} / {len(episodes)} episodes pressed")

## Summary

In [ ]:
summary = episodes.groupby("team_name").agg(
    n_episodes=("episode_id", "count"),
    pct_pressed=("pressed", "mean"),
    mean_pct_time_pressed=("pct_time_pressed", "mean"),
    median_time_to_first_press_sec=("time_to_first_press_sec", "median"),
    mean_duration_sec=("duration_sec", "mean"),
).round(3)
print("by build-up team:")
display(summary)

print("outcome by pressed / unpressed (count):")
ct = pd.crosstab([episodes["team_name"], episodes["pressed"]], episodes["outcome"])
display(ct)
print("outcome share within each group:")
display(ct.div(ct.sum(axis=1), axis=0).round(2))

print("what drives the pressure frames inside pressed episodes (share of pressure frames):")
trig = episodes.groupby("team_name")[["raw_frames_prox_only", "raw_frames_lane_only", "raw_frames_both"]].sum()
display(trig.div(trig.sum(axis=1), axis=0).round(2))

display(episodes.head(10))

## Timeline

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3))
_ids = sorted(TEAM_NAMES)
for r in episodes.itertuples():
    ax.broken_barh([(r.start_frame / FPS / 60, r.n_frames / FPS / 60)], (r.team - 0.35, 0.7),
                   facecolors="#cb3524" if r.pressed else "#9aa0a6")
ax.set_yticks(_ids)
ax.set_yticklabels([TEAM_NAMES[t] for t in _ids])
ax.set_xlabel("match time (min)")
ax.set_title("Build-up episodes (red = pressed, grey = not pressed)")
plt.tight_layout()
plt.show()

## Save

In [ ]:
out_path = CACHE / "build_up_episodes.parquet"
episodes.to_parquet(out_path, index=False)
print("saved:", out_path, f"({len(episodes)} rows)")